<a href="https://colab.research.google.com/github/patriciarrs/RAG/blob/main/PatriciaSilva_RAG_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Final Project Information - RAG [don't edit this section]

## IMPORTANT
**Add comments to your code whenever necessary to explain your logic.**

## Submission
- Single Deliverable: Google Colab Notebook
- File Name Format: [YourName]_RAG_FinalProject.ipynb (optional but it helps us track your file)
- Submit as: ZIP file containing only the .ipynb file (the student portal doesn't accpet .ipynb files)

## Mandatory
- Ingestion Phase
- Inference Phase

# 2. Project Specific Information

Add all the information you find useful to explain your project, the steps you took, the features implemented, and possible enhancements for the future.

## Project Description
[2-3 sentences explaining what your RAG system does and what documents it works with]

## Steps
1. Data collection and preprocessing
2. ...

## Features Implemented
- [x] RAG v2 baseline (mandatory)
- [ ] < additional feature >

# 3. Installation

In [ ]:
%pip install "langchain==0.3.27" -qqq
%pip install "langchain-community==0.3.31" -qqq
%pip install "langchain-openai==0.3.35" -qqq
%pip install "langchain-pinecone==0.2.0" -qqq
%pip install pypdf -qqq
%pip install flashrank -qqq
%pip install ragas -qqq

!pip install -q unstructured layoutparser torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 24.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 4. Setup

In [ ]:
def load_secrets(keys):
    import os

    try:
        from google.colab import userdata  # type: ignore
        values = {key: userdata.get(key) for key in keys}
        in_colab = True
    except Exception:
        # Load from .env file for local development
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            raise ImportError(
                "python-dotenv is required for local development. "
                "Install it with: pip install python-dotenv"
            )

        # Get values from environment (loaded from .env file)
        values = {key: os.getenv(key) for key in keys}
        in_colab = False

    missing = [key for key, value in values.items() if not value]
    if missing:
        if in_colab:
            raise ValueError(f"Missing keys: {', '.join(missing)}. Set them in Colab Secrets.")
        else:
            env_file_help = (
                f"Missing keys: {', '.join(missing)}.\n\n"
                "Create a .env file in the project root with:\n"
                + "\n".join([f"{key}=YOUR_VALUE_HERE" for key in missing])
            )
            raise ValueError(env_file_help)

    for key, value in values.items():
        os.environ[key] = value

    return values

In [ ]:
try:
    secrets = load_secrets(['PINECONE_API_KEY', 'OPENAI_API_KEY'])
except ValueError as e:
    print(e)

# 5. Global Variables

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter

INDEX_NAME = "rag"

# Embeddings Model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

# Classification LLM
classification_llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

# Vectorstore
vectorstore = PineconeVectorStore(
    embedding=embeddings,
    index_name=INDEX_NAME
)

# Parent splitter (large chunks for context)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,   # Large chunks for rich context in answers
    chunk_overlap=400, # 20% overlap
)

# Child splitter (small chunks for search)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,   # Small chunks for precise search matching
    chunk_overlap=75, # 15% overlap
)

# InMemoryStore for parent documents
parent_docstore = InMemoryStore()

# ParentDocumentRetriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=parent_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

# 6. Indexing / Ingestion Phase

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Load documents (data)
# Chunk documents
# Create embeddings
# Store in vector database

## Load documents (data) Function

In [ ]:
import requests
from langchain_community.document_loaders import UnstructuredURLLoader

def load_public_github_markdown_dir(owner, repo, path, branch="main"):
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={branch}"
    response = requests.get(api_url)

    if response.status_code != 200:
        raise Exception(f"Failed to fetch directory contents. Check your repository details or path. Status: {response.status_code}")

    contents = response.json()

    raw_urls = [
        item["download_url"]
        for item in contents
        if item["type"] == "file" and item["name"].endswith(".md")
    ]

    if not raw_urls:
        print("No markdown files found in the specified directory.")
        return []

    print(f"Found {len(raw_urls)} markdown files. Downloading and parsing...")

    loader = UnstructuredURLLoader(urls=raw_urls)
    documents = loader.load()

    return documents

## Topic Detection Function

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def detect_document_topic(document) -> str:
    """
    DETECT DOCUMENT TOPIC

    Args:
        documents (list): List of Document objects from PyPDFLoader

    Returns:
        str: Detected topic (e.g., 'runbooks', 'ADRs')
    """

    topic_detection_template = ChatPromptTemplate.from_template(
      """
      Analyze the following document content and determine its primary topic.

      Document content:
      {content}

      Based on this content, what is the primary topic? Answer with a single word or short phrase (e.g., 'runbooks', 'ADRs').

      Examples:
      If the document is about Engineering Runbooks, answer: runbooks
      If the document is about Architecture Decision Records, answer: ADRs
      If the document is about Onboarding, answer: Onboarding
      If the document is about Company Policies & Standards, answer: Policies & Standards
      If the document is about System Reference, answer: System Reference

      Primary topic:
      """
    )

    topic_detection_chain = topic_detection_template | classification_llm | StrOutputParser()

    # Extract sample content from first few pages
    sample_content = ""
    for doc in documents[:3]:
        sample_content += doc.page_content + "\n\n"

    # Limit to 4000 characters to save costs
    sample_content = sample_content[:4000]

    detected_topic = topic_detection_chain.invoke({
        "content": sample_content
    }).strip().lower()

    return detected_topic

## Clean Text Function

In [ ]:
import re

def clean_markdown_for_rag(text: str) -> str:
    # 1. Strip YAML frontmatter (text between the first pair of --- at the start)
    text = re.sub(r'^---\s*\n.*?\n---\s*\n', '', text, flags=re.DOTALL)

    # 2. Clean Markdown Images: Keep the alt text description, drop the URL noise
    # Transforms ![Alt Text](http://url.com/image.png) -> Image: Alt Text
    text = re.sub(r'!\[(.*?)\]\((.*?)\)', r'Image: \1', text)

    # 3. Clean Hyperlinks: Keep the anchor text, drop or preserve the URL cleanly
    # Transforms [Anchor Text](http://url.com) -> Anchor Text
    text = re.sub(r'\[(.*?)\]\((.*?)\)', r'\1', text)

    # 4. Remove raw inline HTML tags (e.g., <br>, <div class="...">)
    text = re.sub(r'<[^>]*>', '', text)

    # 5. Collapse excessive consecutive blank lines down to a maximum of two
    # (Preserves paragraph breaks for the structural text splitters)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

## Ingest Function

In [ ]:
from langchain.document_loaders import PyPDFLoader

def ingest_document(document):
    # Auto-detect topic
    print(f"\n[2/5] Auto-detecting topic...")
    detected_topic = detect_document_topic(document)
    print(f"✓ Topic: '{detected_topic}'")

    # Preprocessing
    print(f"\n[3/5] Preprocessing...")
    clean_markdown_for_rag(document)
    print(f"✓ Cleaned {len(document)} pages")

    # Add metadata
    print(f"\n[4/5] Adding metadata...")
    for page in document:
        page.metadata.update({'topic': detected_topic})
    print(f"✓ Metadata added")

    # Store in BOTH vectorstores
    print(f"\n[5/5] Storing in vectorstores...")

    # 1. ParentDocumentRetriever (for parent-child strategies)
    parent_retriever.add_documents(document)

    print(f"  ✓ Stored in parent-child vectorstore ({len(document)} documents)")

    # 2. Regular chunks (for baseline + multi-query)
    # chunk_size=800: Slightly smaller than v1-v3 for evaluation comparison
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,   # ~1000 tokens per chunk
        chunk_overlap=200, # 20% overlap
    )

    chunks = splitter.split_documents(document)

    vectorstore.add_documents(chunks)

    print(f"  ✓ Stored in main vectorstore ({len(chunks)} chunks)")

    print(f"\n✓ Ingestion complete!")

    return len(document), detected_topic

In [ ]:
from langchain.document_loaders import PyPDFLoader

def ingest_documents(path: str):
    """
    INGESTION FOR EVALUATION

    Stores documents in vectorstore
    """

    print("-" * 80)
    print(f"STARTING INGESTION - VERSION {VERSION}")
    print("-" * 80)

    # Load documents
    print("\n[1/5] Loading files...")
    REPO_OWNER = "patriciarrs"
    REPO_NAME = "RAG"
    documents = load_public_github_markdown_dir(REPO_OWNER, REPO_NAME, path)
    print(f"Successfully loaded {len(documents)} documents into memory!")

    for document in documents:
      ingest_document(document)

    # return len(documents), detected_topic

## Ingest

In [ ]:
ingest_documents("knowledge_base")

# 7. Inference Phase (RAG)

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Retriever setup (if not declared during setup)
# Prompt template
# LLM integration
# LCEL chain creation

# 8. User Interface (Gradio or other) - optional

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Gradio UI implementation
# Query processing function
# Launch interface

# 9. Testing / Demo

In [ ]:
# =========================================
# Example Steps:
# =========================================
# Example Queries to Test
# Functions to Test
# Demo information